In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

In [4]:
BASE_URL = "https://www.storia.ro/ro/rezultate/vanzare/apartament/iasi/iasi"
NUMAR_PAGINI = 30
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9,ro;q=0.8"
}


def fetch_complete_listing(url: str) -> str:
    """
    Upgraded scraping: now we get the whole description from each listing
    None if we cant extract it 
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            print(f"  [WARN] inaccessible ({resp.status_code}): {url}")
            return None

        soup = BeautifulSoup(resp.content, "html.parser")

        selectori = [
            {"data-cy": "advert-description"},
            {"data-testid": "description"},
            {"class": lambda c: c and "description" in " ".join(c).lower()},
        ]

        for selector in selectori:
            tag = soup.find("div", selector)
            if tag:
                return tag.get_text(separator=" ", strip=True)

        for p in soup.find_all("p"):
            text = p.get_text(strip=True)
            if len(text) > 100: 
                return text

        return None

    except Exception as e:
        print(f"  [ERROR] fetch_complete_listing: {e}")
        return None


data_colectata = []

for pagina in range(1, NUMAR_PAGINI + 1):
    url_curent = f"{BASE_URL}?page={pagina}"
    print(f"\nPagina {pagina}: {url_curent} ...")

    try:
        response = requests.get(url_curent, headers=HEADERS, timeout=15)

        if response.status_code != 200:
            print(f"Eroare la pagina {pagina} (Cod: {response.status_code})")
            continue

        soup = BeautifulSoup(response.content, "html.parser")
        anunturi = soup.find_all("article")

        if not anunturi:
            print("No more listings. Stop.")
            break

        for card in anunturi:
            item = {}

            titlu_tag = card.find("h3")
            if not titlu_tag:
                titlu_tag = card.find("span", {"data-cy": "listing-item-title"})
            item["Titlu"] = titlu_tag.get_text(strip=True) if titlu_tag else "Titlu Necunoscut"

            pret_text = "N/A"
            for span in card.find_all("span"):
                text = span.get_text(strip=True)
                if "€" in text:
                    pret_text = text
                    break
            item["Pret_Raw"] = pret_text

            link_tag = card.find("a", href=True)
            link = "https://www.storia.ro" + link_tag["href"] if link_tag else "N/A"
            item["Link"] = link

            item["Descriere_Preview"] = card.get_text(separator=" | ", strip=True)

            if link != "N/A":
                print(f"  Fetch descriere: {link}")
                item["Descriere_Completa"] = fetch_complete_listing(link)
                time.sleep(random.uniform(1.5, 3.0))
            else:
                item["Descriere_Completa"] = None

            data_colectata.append(item)

        print(f"Pagina {pagina} finalizata. Total anunturi: {len(data_colectata)}")
        time.sleep(random.uniform(3, 5))

    except Exception as e:
        print(f"Eroare la pagina {pagina}: {e}")

df = pd.DataFrame(data_colectata)


def curata_pret(p):
    if pd.isna(p) or p == "N/A":
        return None
    clean = p.replace(" ", "").replace("€", "").replace(",", ".")
    try:
        return float(clean)
    except ValueError:
        return None


df["Pret_EUR"] = df["Pret_Raw"].apply(curata_pret)

print("\n--- First listings ---")
print(df[["Titlu", "Pret_EUR", "Link", "Descriere_Completa"]].head())

nume_fisier = "3_imobiliare_iasi_raw.csv"
df.to_csv(nume_fisier, index=False, encoding="utf-8-sig")
print(f"\nDate salvate în {nume_fisier} ({len(df)} anunțuri)")


Pagina 1: https://www.storia.ro/ro/rezultate/vanzare/apartament/iasi/iasi?page=1 ...
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-cu-2-camere-mobilat-si-utilat-loc-de-parcare-subteran-IDEBD8
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-3-camere-mobilat-etaj-48-bloc-nou-palas-IDGsZ5
  Fetch descriere: https://www.storia.ro/ro/oferta/2-camere-etaj-1-70mp-bloc-intabulat-loc-parcare-copou-hotel-grand-view-IDGsYA
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-3-camere-centru-128mp-vedere-panoramica-IDGsYs
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-200-m-iasi-judet-iasi-IDFnyE
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-2-camere-61-mp-bucium-IDFD2M
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-2-camere-60-mp-zona-cug-complex-adamant-towers-IDGg9c
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-2-camere-57-mp-complex-bucium-confort-IDG1Ng
  Fetch descriere: https://www.storia.r

### OBS:

-   Pentru 30 pagini, aprox 1100 anunturi a durat undeva la 54 minute (o ora sa zicem)

### Probleme:

Prima iteratie:

-   "Conectează-te sau creează un cont pentru a accesa istoricul complet al anunțurilor, inclusiv modificările de preț", nu am putut lua intreaga descriere



In [5]:
import json

In [ ]:
BASE_URL = "https://www.storia.ro/ro/rezultate/vanzare/apartament/iasi/iasi"
NUMAR_PAGINI = 1
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9,ro;q=0.8"
}

def fetch_complete_listing(url: str) -> str:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            print(f"  [WARN] {resp.status_code}: {url}")
            return None

        soup = BeautifulSoup(resp.content, "html.parser")

        next_data_tag = soup.find("script", {"id": "__NEXT_DATA__"})
        if next_data_tag:
            try:
                data = json.loads(next_data_tag.string)
                with open("debug_next_data.json", "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)
                print("DEBUG: salvat debug_next_data.json")
                exit()
                props = data.get("props", {})
                page_props = props.get("pageProps", {})

                descriere = (
                    # Cale 1
                    page_props.get("ad", {}).get("description")
                    # Cale 2
                    or page_props.get("advert", {}).get("description")
                    # Cale 3 - uneori e nested mai adânc
                    or page_props.get("data", {}).get("advert", {}).get("description")
                )

                if descriere:
                    return descriere.strip()

            except (json.JSONDecodeError, AttributeError) as e:
                print(f"  [WARN] JSON parse error: {e}")

        # ── Metoda 2: Fallback pe data-cy / clase CSS ────────────────────────
        for selector in [
            {"data-cy": "advert-description"},
            {"data-testid": "description"},
        ]:
            tag = soup.find("div", selector)
            if tag:
                return tag.get_text(separator=" ", strip=True)

        return None

    except Exception as e:
        print(f"  [ERROR] {e}")
        return None


data_colectata = []

for pagina in range(1, NUMAR_PAGINI + 1):
    url_curent = f"{BASE_URL}?page={pagina}"
    print(f"\nPagina {pagina}: {url_curent} ...")

    try:
        response = requests.get(url_curent, headers=HEADERS, timeout=15)

        if response.status_code != 200:
            print(f"Eroare la pagina {pagina} (Cod: {response.status_code})")
            continue

        soup = BeautifulSoup(response.content, "html.parser")
        anunturi = soup.find_all("article")

        if not anunturi:
            print("No more listings. Stop.")
            break

        for card in anunturi:
            item = {}

            titlu_tag = card.find("h3")
            if not titlu_tag:
                titlu_tag = card.find("span", {"data-cy": "listing-item-title"})
            item["Titlu"] = titlu_tag.get_text(strip=True) if titlu_tag else "Titlu Necunoscut"

            pret_text = "N/A"
            for span in card.find_all("span"):
                text = span.get_text(strip=True)
                if "€" in text:
                    pret_text = text
                    break
            item["Pret_Raw"] = pret_text

            link_tag = card.find("a", href=True)
            link = "https://www.storia.ro" + link_tag["href"] if link_tag else "N/A"
            item["Link"] = link

            item["Descriere_Preview"] = card.get_text(separator=" | ", strip=True)

            if link != "N/A":
                print(f"  Fetch descriere: {link}")
                item["Descriere_Completa"] = fetch_complete_listing(link)
                time.sleep(random.uniform(1.5, 3.0))
            else:
                item["Descriere_Completa"] = None

            data_colectata.append(item)

        print(f"Pagina {pagina} finalizata. Total anunturi: {len(data_colectata)}")
        time.sleep(random.uniform(3, 5))

    except Exception as e:
        print(f"Eroare la pagina {pagina}: {e}")

df = pd.DataFrame(data_colectata)


def curata_pret(p):
    if pd.isna(p) or p == "N/A":
        return None
    clean = p.replace(" ", "").replace("€", "").replace(",", ".")
    try:
        return float(clean)
    except ValueError:
        return None


df["Pret_EUR"] = df["Pret_Raw"].apply(curata_pret)

print("\n--- First listings ---")
print(df[["Titlu", "Pret_EUR", "Link", "Descriere_Completa"]].head())

nume_fisier = "3_imobiliare_iasi_raw.csv"
df.to_csv(nume_fisier, index=False, encoding="utf-8-sig")
print(f"\nDate salvate în {nume_fisier} ({len(df)} anunțuri)")


Pagina 1: https://www.storia.ro/ro/rezultate/vanzare/apartament/iasi/iasi?page=1 ...
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-cu-3-camere-copou-IDGt3F
DEBUG: salvat debug_next_data.json
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-de-vanzare-cu-2-camere-decomandat-zona-visani-IDEPFI
DEBUG: salvat debug_next_data.json
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-cu-doua-camere-copou-IDGt3c
DEBUG: salvat debug_next_data.json
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-2-camere-de-vanzare-podu-ros-IDGt2X
DEBUG: salvat debug_next_data.json
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-2-camere-de-vanzare-bucium-IDGt2V
DEBUG: salvat debug_next_data.json
  Fetch descriere: https://www.storia.ro/ro/oferta/2-camere-etaj-intermediar-76mp-bloc-nou-family-market-IDGt2R
DEBUG: salvat debug_next_data.json
  Fetch descriere: https://www.storia.ro/ro/oferta/apartament-3-camere-100-mp-lidl-lunca-cetatuii-IDFw8c
D

: 

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import json
import re
import html as html_module

In [3]:
BASE_URL = "https://www.storia.ro/ro/rezultate/vanzare/apartament/iasi/iasi"
NUMAR_PAGINI = 30
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9,ro;q=0.8"
}


def clean_html(raw: str) -> str:
    """
    Elimină tag-urile HTML și decodifică entitățile HTML (e.g. &amp; → &, &#259; → ă).
    """
    if not raw:
        return None
    # Decodifică entități HTML (&amp; &acirc; &#259; etc.)
    text = html_module.unescape(raw)
    # Elimină tag-urile HTML rămase (<p>, <br/>, <b> etc.)
    soup = BeautifulSoup(text, "html.parser")
    # Înlocuiește <br> și <p> cu newline pentru lizibilitate
    for tag in soup.find_all(["br", "p"]):
        tag.replace_with("\n" + tag.get_text())
    return re.sub(r'\n{3,}', '\n\n', soup.get_text()).strip()


def curata_pret(p) -> float | None:
    """
    Curăță prețul: gestionează spații normale, non-breaking (\\xa0, \\u202f),
    puncte/virgule ca separatori de mii și returnează float.
    """
    if pd.isna(p) or p == "N/A":
        return None
    # Elimină ORICE tip de spațiu (normal, non-breaking, narrow no-break etc.)
    clean = re.sub(r'[\s\u00a0\u202f\u2009\u2007\u2008]', '', str(p))
    clean = clean.replace('€', '').replace(',', '.')
    # Dacă sunt mai multe puncte (ex: 1.177.270), păstrăm doar cifrele
    parts = clean.split('.')
    if len(parts) > 2:
        clean = ''.join(parts)
    try:
        return float(clean)
    except ValueError:
        return None


def fetch_listing_data(url: str) -> dict:
    """
    Accesează pagina unui anunț individual și extrage:
    - titlu complet
    - descriere completă (text curat, fără HTML)

    Strategia principală: __NEXT_DATA__ JSON (Next.js)
    Fallback: selectori CSS/data-cy
    """
    result = {"titlu": None, "descriere": None}

    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            print(f"  [WARN] {resp.status_code}: {url}")
            return result

        soup = BeautifulSoup(resp.content, "html.parser")

        # ── Metoda 1: __NEXT_DATA__ JSON ────────────────────────────────────
        next_data_tag = soup.find("script", {"id": "__NEXT_DATA__"})
        if next_data_tag:
            try:
                data = json.loads(next_data_tag.string)
                page_props = data.get("props", {}).get("pageProps", {})

                # Căi posibile pentru obiectul anunțului în JSON-ul storia.ro
                ad_obj = (
                    page_props.get("ad")
                    or page_props.get("advert")
                    or page_props.get("data", {}).get("advert")
                    or page_props.get("data", {}).get("ad")
                )

                if ad_obj:
                    result["titlu"] = ad_obj.get("title") or ad_obj.get("name")
                    raw_desc = ad_obj.get("description") or ad_obj.get("body")
                    if raw_desc:
                        result["descriere"] = clean_html(raw_desc)
                        return result

            except (json.JSONDecodeError, AttributeError) as e:
                print(f"  [WARN] JSON parse error: {e}")

        # ── Metoda 2: Fallback selectori HTML ───────────────────────────────
        # Titlu
        for sel in [{"data-cy": "ad-title"}, {"data-cy": "listing-item-title"}]:
            tag = soup.find(["h1", "h2", "h3"], sel)
            if tag:
                result["titlu"] = tag.get_text(strip=True)
                break
        if not result["titlu"]:
            h1 = soup.find("h1")
            if h1:
                result["titlu"] = h1.get_text(strip=True)

        # Descriere
        for sel in [{"data-cy": "advert-description"}, {"data-testid": "description"}]:
            tag = soup.find("div", sel)
            if tag:
                result["descriere"] = clean_html(str(tag))
                return result

        # Fallback generic: primul paragraf lung
        for p in soup.find_all("p"):
            text = p.get_text(strip=True)
            if len(text) > 100:
                result["descriere"] = html_module.unescape(text)
                break

        return result

    except Exception as e:
        print(f"  [ERROR] {e}")
        return result


# ─────────────────────────────────────────────────────────────────────────────
data_colectata = []

for pagina in range(1, NUMAR_PAGINI + 1):
    url_curent = f"{BASE_URL}?page={pagina}"
    print(f"\nPagina {pagina}: {url_curent} ...")

    try:
        response = requests.get(url_curent, headers=HEADERS, timeout=15)

        if response.status_code != 200:
            print(f"Eroare la pagina {pagina} (Cod: {response.status_code})")
            continue

        soup = BeautifulSoup(response.content, "html.parser")
        anunturi = soup.find_all("article")

        if not anunturi:
            print("Nu s-au mai găsit anunțuri. Oprire.")
            break

        for card in anunturi:
            item = {}

            # --- Preț (din card, înainte de fetch individual) ---
            pret_text = "N/A"
            for span in card.find_all("span"):
                text = span.get_text(strip=True)
                if "€" in text and any(c.isdigit() for c in text):
                    pret_text = text
                    break
            item["Pret_Raw"] = pret_text

            # --- Link ---
            link_tag = card.find("a", href=True)
            link = "https://www.storia.ro" + link_tag["href"] if link_tag else "N/A"
            # Ignoră URL-urile /hpr/ (promoted duplicates)
            if "/hpr/" in link:
                continue
            item["Link"] = link

            # --- Descriere preview (din card) ---
            item["Descriere_Preview"] = card.get_text(separator=" | ", strip=True)

            # --- Fetch pagina individuală: titlu + descriere completă ---
            if link != "N/A":
                print(f"  Fetch: {link}")
                listing_data = fetch_listing_data(link)
                item["Titlu"] = listing_data["titlu"] or "Titlu Necunoscut"
                item["Descriere_Completa"] = listing_data["descriere"]
                time.sleep(random.uniform(1.5, 3.0))
            else:
                item["Titlu"] = "Titlu Necunoscut"
                item["Descriere_Completa"] = None

            data_colectata.append(item)

        print(f"Pagina {pagina} finalizată. Total: {len(data_colectata)} anunțuri")
        time.sleep(random.uniform(3, 5))

    except Exception as e:
        print(f"Eroare la pagina {pagina}: {e}")


# ─────────────────────────────────────────────────────────────────────────────
df = pd.DataFrame(data_colectata)

df["Pret_EUR"] = df["Pret_Raw"].apply(curata_pret)

# Reordonare coloane
cols = ["Titlu", "Pret_EUR", "Pret_Raw", "Link", "Descriere_Completa", "Descriere_Preview"]
df = df[[c for c in cols if c in df.columns]]

print("\n--- Primele înregistrări ---")
print(df[["Titlu", "Pret_EUR", "Descriere_Completa"]].head(3).to_string())

nume_fisier = "4_imobiliare_iasi_raw.csv"
df.to_csv(nume_fisier, index=False, encoding="utf-8-sig")
print(f"\nSalvat: {nume_fisier} ({len(df)} anunțuri)")


Pagina 1: https://www.storia.ro/ro/rezultate/vanzare/apartament/iasi/iasi?page=1 ...
  Fetch: https://www.storia.ro/ro/oferta/apartament-5-camere-135-mp-terasa-250-mp-zona-visani-IDFWez
  Fetch: https://www.storia.ro/ro/oferta/apartament-2-camere-decomandat-tatarasi-dispecer-IDGt9Q
  Fetch: https://www.storia.ro/ro/oferta/apartament-decomandat-3-camere-2-bai-zona-nicolina-IDFbOO
  Fetch: https://www.storia.ro/ro/oferta/direct-dezvoltator-ap-1cam-etaj-1-balcon-loc-parcare-IDGt8W
  Fetch: https://www.storia.ro/ro/oferta/ap-2cam-etaj-1-mobilat-ttrai-apropiere-tudor-IDGt8t
  Fetch: https://www.storia.ro/ro/oferta/bularga-garsoniera-17-mp-etaj-1-renovata-complet-bucatarie-pe-com-IDGt8n
  Fetch: https://www.storia.ro/ro/oferta/apartament-cu-doua-camere-palas-campus-centru-iasi-ac-parcare-IDGt7I
  Fetch: https://www.storia.ro/ro/oferta/apartament-2-camere-decomandat-complex-royal-town-IDxeHb
  Fetch: https://www.storia.ro/ro/oferta/predare-aprilie-apartament-1-camer-44-mp-royal-town-iai-IDxe

### OBS

-   Pentru 30 de pagini, aprox 1060 anunturi a durat 53 de minute

-   De data asta datele sunt extrase foarte bine, am descrierile complete